# 🎬 RÉALITÉ TOTALE 2026 — Serveur API Gradio
**CPU | Tunnel public .gradio.live**

1. Exécute toutes les cellules dans l'ordre
2. Copie le lien `https://xxxx.gradio.live` qui apparaît à la fin
3. Colle ce lien dans ton fichier HTML

In [ ]:
# CELLULE 1 — Installation
!pip install -q gradio==4.44.0 diffusers transformers accelerate \
    Pillow torch torchvision huggingface-hub safetensors einops
print('✅ Installation terminée')

In [ ]:
# CELLULE 2 — HF_TOKEN
# ⚠️  IMPORTANT : Dans la barre gauche 🔑 Secrets,
#     active le bouton "Accès au notebook" pour HF_TOKEN
import os

HF_TOKEN = ''
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('✅ HF_TOKEN chargé depuis les Secrets Colab')
except Exception as e:
    print(f'⚠️  Erreur Secrets : {e}')

# Si HF_TOKEN est vide, colle ton token ici :
if not HF_TOKEN:
    HF_TOKEN = ''  # ← colle ton token entre les guillemets

if not HF_TOKEN:
    raise ValueError('❌ HF_TOKEN manquant ! Colle ton token dans la ligne ci-dessus.')

os.environ['HF_TOKEN'] = HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print('✅ Connecté à HuggingFace')

In [ ]:
# CELLULE 3 — Cloner le dépôt
import os
if not os.path.exists('/content/newaivideo'):
    !git clone https://github.com/avenircc120-debug/newaivideo /content/newaivideo
else:
    !git -C /content/newaivideo pull
os.chdir('/content/newaivideo')
print('✅ Dépôt prêt')

In [ ]:
# CELLULE 4 — Structure Maître
import sys
sys.path.insert(0, '/content/newaivideo')
from mount_bridge import demarrer_systeme
systeme = demarrer_systeme()
CHEMIN_VITESSE = systeme['chemins']['vitesse']
print(f'✅ Sortie vidéo → {CHEMIN_VITESSE}')

In [ ]:
# CELLULE 5 — Moteur
import sys, os, json, time
from pathlib import Path
from PIL import Image
import numpy as np
sys.path.insert(0, '/content/newaivideo')
from prompt_interpreter import interpret_user_order
from points_matrix import build_2000_points_matrix
from pipeline import RealityPipeline

def generer_video(photo_numpy, description):
    if photo_numpy is None:
        return None, '❌ Chargez votre photo.'
    if not description or not description.strip():
        return None, '❌ Écrivez une description.'
    debut = time.time()
    try:
        face_image = Image.fromarray(photo_numpy.astype('uint8')).convert('RGB')
        ordre  = interpret_user_order(description)
        matrix = build_2000_points_matrix(overrides=ordre['point_overrides'])
        runner = RealityPipeline(
            face_image      = face_image,
            prompt          = ordre['enriched_prompt'],
            tools_activated = ordre['tools_activated'],
            matrix          = matrix,
            output_dir      = CHEMIN_VITESSE,
            resolution      = (1920, 1080),
            num_frames      = 72,
            fps             = 24,
            use_gpu         = False,
        )
        result     = runner.run()
        duree      = time.time() - debut
        video_path = Path(CHEMIN_VITESSE) / 'video_finale.mp4'
        rapport = (
            f'✅ Généré en {duree:.1f}s\n'
            f'Scènes  : {", ".join(ordre["scene_labels"]) or "Standard"}\n'
            f'Outils  : {len(result["tools_ran"])}/14\n'
            f'Points  : {result["final_params"].get("total_points_injected", 0)}/2000\n'
            f'Sortie  : {video_path}'
        )
        return str(video_path) if result['success'] else None, rapport
    except Exception as e:
        return None, f'❌ Erreur : {str(e)}'

print('✅ Moteur prêt')

In [ ]:
# CELLULE 6 — Interface Gradio + Lien public
import gradio as gr

with gr.Blocks(title='Réalité Totale 2026') as demo:
    gr.Markdown('# 🎬 Réalité Totale 2026 — La Grande Chose')
    gr.Markdown('Pipeline 14 outils IA · 2000 points · 4K Arri Alexa')

    with gr.Row():
        with gr.Column():
            photo_input = gr.Image(
                label  = '📸 Votre photo (visage)',
                type   = 'numpy',
                height = 300,
            )
            texte_input = gr.Textbox(
                label       = '✍️ Description de la scène',
                placeholder = 'Ex: Moi, marchant à la foire de Cotonou, au soleil couchant.',
                lines       = 4,
            )
            btn = gr.Button('🚀 GÉNÉRER LA VIDÉO', variant='primary')

        with gr.Column():
            video_out  = gr.Video(label='🎬 Vidéo générée')
            rapport_out= gr.Textbox(label='📊 Rapport', lines=7)

    btn.click(
        fn      = generer_video,
        inputs  = [photo_input, texte_input],
        outputs = [video_out, rapport_out],
    )

# ⚡ Lancement — le lien .gradio.live apparaît juste en dessous
demo.launch(share=True, server_port=7860, show_error=True)